In [ ]:
import os
import numpy as np
# 设置 Hugging Face 镜像（如需要）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from vllm import LLM, SamplingParams

# ============================================================
# 第一步：准备知识库文档
# ============================================================
from charset_normalizer import from_path

def load_txt_lines(file_path):
    result = from_path(file_path).best()
    enc = result.encoding if result else None

    if enc:
        try:
            print(f"识别编码{enc}")
            with open(file_path, 'r', encoding=enc) as f:
                documents = f.read()
            return documents
        except UnicodeDecodeError:
            raise OSError(f"识别编码{enc}错误，文件解码失败")
    else:
        raise OSError("无法确定文件编码")

file_path = '../data/txt/西游记+上卷.txt'
documents = load_txt_lines(file_path)

INFO 07-23 14:32:19 __init__.py:190] Automatically detected platform cuda.
识别编码gb18030


In [4]:
new_documents = documents.replace("\n", " ").replace("\r", " ").split("。")  # 按句号分割为句子列表
page_doc = []
for raw in new_documents:
    page_doc.append(raw.strip() + "。")  # 添加句号并去除首尾空格
print(f"文档总句子数: {len(page_doc)}")
print(page_doc[50:55])

文档总句子数: 10730
['盖自开辟以来，每受天真地秀，日精月华，感之既久，遂有灵通之意。', '内育仙胞，一日迸裂，产一石卵，似圆球样大。', '因见风，化作一个石猴，五官俱备，四肢皆全。', '便就学爬学走，拜了四方。', '目运两道金光，射冲斗府。']


In [ ]:

# ============================================================
# 第二步：加载嵌入模型，为文档生成向量
# ============================================================
print("正在加载嵌入模型 intfloat/e5-small ...")
# vLLM 0.7.2 用 task="embed" 来加载嵌入模型
try:
    embedding_llm = LLM(
        model="intfloat/e5-small",
        task="embed",              # 指定嵌入任务
        enforce_eager=True
    )
    print("嵌入模型加载成功")
except Exception as e:
    print(f"加载嵌入模型失败: {e}")
    print("尝试降级方案：使用 sentence-transformers ...")
    # 降级方案：使用 sentence-transformers
    from sentence_transformers import SentenceTransformer
    # 创建降级方案的替代函数（见下方）

# 为文档生成嵌入向量
print("正在生成文档向量...")
# e5-small 要求文本前加 "passage: " 前缀
doc_texts = [f"passage: {doc}" for doc in page_doc]
doc_outputs = embedding_llm.embed(doc_texts)
doc_embeddings = np.array([output.outputs.embedding for output in doc_outputs])
print(f"文档向量维度: {doc_embeddings.shape}")  # (10, 384)

In [ ]:


# ============================================================
# 第三步：实现简单的向量检索
# ============================================================
def retrieve(query, embeddings, texts, k=3):
    """
    计算 query 与所有文档的余弦相似度，返回最相关的 k 个文档
    """
    # 为查询生成向量（e5-small 要求 "query: " 前缀）
    query_output = embedding_llm.embed([f"query: {query}"])
    query_embedding = np.array(query_output[0].outputs.embedding)

    # 计算余弦相似度
    # 先归一化
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    doc_norms = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    # 点积 = 余弦相似度（已归一化）
    similarities = np.dot(query_norm, doc_norms.T)

    # 取最相似的 k 个
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_docs = [texts[i] for i in top_k_indices]
    top_k_scores = similarities[top_k_indices]

    return top_k_docs, top_k_scores


# ============================================================
# 第四步：加载生成模型
# ============================================================
print("\n正在加载生成模型 Qwen2.5-1.5B-Instruct ...")
gen_llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    gpu_memory_utilization=0.7,
    max_model_len=1024,
    enforce_eager=True
)
print("生成模型加载成功")


# ============================================================
# 第五步：构建 RAG 问答函数
# ============================================================
def rag_answer(query, k=3):
    """
    检索相关文档，拼接成 prompt，生成答案
    """
    # 检索
    retrieved_docs, scores = retrieve(query, doc_embeddings, documents, k=k)

    # 构建 RAG prompt
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])
    rag_prompt = f"""你是一个知识渊博的助手。请根据以下参考资料回答问题。
如果参考资料不足以回答，请如实说明。

【参考资料】
{context}

【问题】
{query}

【回答】"""

    # 生成
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=256
    )
    output = gen_llm.generate([rag_prompt], sampling_params)

    return {
        "query": query,
        "retrieved_docs": list(zip(retrieved_docs, scores)),
        "answer": output[0].outputs[0].text
    }


# ============================================================
# 第六步：对比实验 —— 有 RAG vs 无 RAG
# ============================================================
def direct_answer(query):
    """
    不用 RAG，直接问模型
    """
    direct_prompt = f"请回答问题：{query}"
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=256
    )
    output = gen_llm.generate([direct_prompt], sampling_params)
    return output[0].outputs[0].text


# ============================================================
# 第七步：运行测试
# ============================================================
test_queries = [
    "什么是深度学习？",
    "Transformer 架构有什么特点？",
    "vLLM 是什么？",
    "谁发明了相对论？",  # 不在知识库中，测试模型是否诚实
]

print("\n" + "="*70)
print("RAG vs 无 RAG 对比实验")
print("="*70)

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"问题: {query}")
    print(f"{'='*70}")

    # RAG 方式
    result = rag_answer(query, k=3)
    print(f"\n[检索到的文档]")
    for doc, score in result["retrieved_docs"]:
        print(f"  [{score:.3f}] {doc}")
    print(f"\n[RAG 回答]")
    print(result["answer"])

    # 直接问答
    direct = direct_answer(query)
    print(f"\n[直接回答]")
    print(direct)

print("\n\n实验完成！请对比以上两种回答的质量和准确性。")